In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("Original Dataset.csv")

print(df.head())
print(df.shape)

                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

In [3]:
print(df["category"].value_counts())

category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [4]:
selected_categories = [
    "POLITICS",
    "ENTERTAINMENT",
    "TRAVEL",
    "PARENTING"
]

df = df[df["category"].isin(selected_categories)]

print(df["category"].value_counts())
print(df.shape)

category
POLITICS         35602
ENTERTAINMENT    17362
TRAVEL            9900
PARENTING         8791
Name: count, dtype: int64
(71655, 6)


In [5]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
link                    0
headline                1
category                0
short_description    6229
authors              9080
date                    0
dtype: int64


In [6]:
print("Duplicate Rows:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Shape after removing duplicates:", df.shape)

Duplicate Rows: 1
Shape after removing duplicates: (71654, 6)


In [7]:
df["text"] = (
    df["headline"].fillna("") + " " +
    df["short_description"].fillna("")
)

df = df[["text", "category"]]

print(df.head())
print(df.shape)

                                                 text       category
3   The Funniest Tweets From Parents This Week (Se...      PARENTING
20  Golden Globes Returning To NBC In January Afte...  ENTERTAINMENT
21  Biden Says U.S. Forces Would Defend Taiwan If ...       POLITICS
24  ‘Beautiful And Sad At The Same Time’: Ukrainia...       POLITICS
28  James Cameron Says He 'Clashed' With Studio Be...  ENTERTAINMENT
(71654, 2)


In [8]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["text"] = df["text"].apply(clean_text)

print(df.head())

                                                 text       category
3   The Funniest Tweets From Parents This Week (Se...      PARENTING
20  Golden Globes Returning To NBC In January Afte...  ENTERTAINMENT
21  Biden Says U.S. Forces Would Defend Taiwan If ...       POLITICS
24  ‘Beautiful And Sad At The Same Time’: Ukrainia...       POLITICS
28  James Cameron Says He 'Clashed' With Studio Be...  ENTERTAINMENT


In [9]:
sample_size = 8791

df = (
    df.groupby("category", group_keys=False)
      .sample(n=sample_size, random_state=42)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print(df["category"].value_counts())
print("Total samples:", len(df))

category
POLITICS         8791
TRAVEL           8791
ENTERTAINMENT    8791
PARENTING        8791
Name: count, dtype: int64
Total samples: 35164


In [10]:
print("Final Dataset Shape:", df.shape)
print("\nCategory Distribution:")
print(df["category"].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Final Dataset Shape: (35164, 2)

Category Distribution:
category
POLITICS         8791
TRAVEL           8791
ENTERTAINMENT    8791
PARENTING        8791
Name: count, dtype: int64

Missing Values:
text        0
category    0
dtype: int64

Duplicate Rows:
32


In [11]:
X = df["text"]
y = df["category"]

print("Input samples:", len(X))
print("Target samples:", len(y))

Input samples: 35164
Target samples: 35164


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 28131
Testing samples: 7033


In [13]:
print("Training Distribution:")
print(y_train.value_counts())

print("\nTesting Distribution:")
print(y_test.value_counts())

Training Distribution:
category
PARENTING        7033
TRAVEL           7033
ENTERTAINMENT    7033
POLITICS         7032
Name: count, dtype: int64

Testing Distribution:
category
POLITICS         1759
ENTERTAINMENT    1758
PARENTING        1758
TRAVEL           1758
Name: count, dtype: int64


In [14]:
!pip install -q transformers datasets accelerate

In [15]:
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [16]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Classes:", label_encoder.classes_)

Classes: ['ENTERTAINMENT' 'PARENTING' 'POLITICS' 'TRAVEL']


In [17]:
train_df = pd.DataFrame({
    "text": X_train.values,
    "label": y_train_encoded
})

test_df = pd.DataFrame({
    "text": X_test.values,
    "label": y_test_encoded
})

print(train_df.head())
print(test_df.head())

                                                text  label
0  For Hanukkah: A Jewish Mom's Christmas Poem No...      1
1  There's No Room For Racism (Or Apathy) In Amer...      2
2  The 10 Types Of People You'll Meet On A Plane ...      3
3  Why Do We Push Our Babies? Rather than pushing...      1
4  My Child's Perfect School Lunch Our ultimate g...      1
                                                text  label
0  Cruz Hits Obama Administration For POW Release...      2
1              Joan Rivers Just Won Her First Grammy      0
2  Home Trampolines For Kids Are Too Dangerous, P...      1
3  Trump's Backdoor Muslim Ban Robert A. Chesnut,...      2
4  Trump White House Staffers Reportedly Have A '...      2


In [18]:
import torch
from torch.utils.data import Dataset

In [19]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [20]:
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Training texts:", len(train_encodings["input_ids"]))
print("Testing texts:", len(test_encodings["input_ids"]))

Training texts: 28131
Testing texts: 7033


In [21]:
class BERTDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        item = {
            key: torch.tensor(value[index])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )

        return item

In [22]:
train_dataset = BERTDataset(
    train_encodings,
    y_train_encoded
)

test_dataset = BERTDataset(
    test_encodings,
    y_test_encoded
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 28131
Testing samples: 7033


In [23]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision = precision_score(
        labels,
        predictions,
        average="weighted"
    )

    recall = recall_score(
        labels,
        predictions,
        average="weighted"
    )

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [24]:
def run_bert_experiment(
    learning_rate,
    batch_size,
    weight_decay
):

    print("\nStarting BERT experiment...")
    print("Learning Rate:", learning_rate)
    print("Batch Size:", batch_size)
    print("Weight Decay:", weight_decay)

    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=4
    )

    num_train_samples = len(train_dataset)

    steps_per_epoch = int(
        np.ceil(num_train_samples / batch_size)
    )

    total_steps = steps_per_epoch * 5

    warmup_steps = int(
        total_steps * 0.1
    )

    print("Warmup Steps:", warmup_steps)

    training_args = TrainingArguments(

        output_dir="./bert_results",

        num_train_epochs=5,

        learning_rate=learning_rate,

        per_device_train_batch_size=batch_size,

        per_device_eval_batch_size=batch_size,

        weight_decay=weight_decay,

        warmup_steps=warmup_steps,

        eval_strategy="epoch",

        save_strategy="epoch",

        load_best_model_at_end=True,

        metric_for_best_model="f1",

        greater_is_better=True,

        logging_strategy="epoch",

        report_to="none"
    )

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        eval_dataset=test_dataset,

        compute_metrics=compute_metrics,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2
            )
        ]
    )

    trainer.train()

    results = trainer.evaluate()

    print("\nFinal Results:")

    print(
        "Accuracy:",
        results["eval_accuracy"]
    )

    print(
        "Precision:",
        results["eval_precision"]
    )

    print(
        "Recall:",
        results["eval_recall"]
    )

    print(
        "F1 Score:",
        results["eval_f1"]
    )

    return results

In [ ]:
results_bert_1 = run_bert_experiment(
    learning_rate=0.00002,
    batch_size=16,
    weight_decay=0.01
)


Starting BERT experiment...
Learning Rate: 2e-05
Batch Size: 16
Weight Decay: 0.01


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warmup Steps: 879


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.399992,0.210731,0.940424,0.941754,0.940424,0.940549
2,0.132623,0.192220,0.952510,0.952512,0.952510,0.952497
3,0.060835,0.239604,0.954500,0.954667,0.954500,0.954515
4,0.026167,0.254669,0.952936,0.953345,0.952936,0.952969
5,0.010364,0.265467,0.956349,0.956465,0.956349,0.956368


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.010364,0.265467,5,0.956349,0.956465,0.956349,0.956368



Final Results:
Accuracy: 0.9563486421157401
Precision: 0.9564647803644236
Recall: 0.9563486421157401
F1 Score: 0.9563678825910042


In [ ]:
results_bert_2 = run_bert_experiment(
    learning_rate=0.00003,
    batch_size=16,
    weight_decay=0.01
)


Starting BERT experiment...
Learning Rate: 3e-05
Batch Size: 16
Weight Decay: 0.01


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warmup Steps: 879


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.369653,0.218563,0.938575,0.940285,0.938575,0.938596
2,0.130135,0.223775,0.949239,0.950088,0.949239,0.949351
3,0.061280,0.229226,0.955496,0.955549,0.955496,0.955462
4,0.025488,0.299353,0.948386,0.949399,0.948386,0.948518
5,0.008398,0.286167,0.957202,0.957207,0.957202,0.957195


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.008398,0.286167,5,0.957202,0.957207,0.957202,0.957195



Final Results:
Accuracy: 0.9572017631167354
Precision: 0.9572068364662504
Recall: 0.9572017631167354
F1 Score: 0.9571953223373034


In [ ]:
results_bert_3 = run_bert_experiment(
    learning_rate=0.00002,
    batch_size=32,
    weight_decay=0.01
)


Starting BERT experiment...
Learning Rate: 2e-05
Batch Size: 32
Weight Decay: 0.01


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.013179,0.216180,5,0.955496,0.955492,0.955496,0.955483



Final Results:
Accuracy: 0.9554955211147448
Precision: 0.9554920942339081
Recall: 0.9554955211147448
F1 Score: 0.9554833262088749


In [ ]:
results_bert_4 = run_bert_experiment(
    learning_rate=0.00003,
    batch_size=32,
    weight_decay=0.01
)


Starting BERT experiment...
Learning Rate: 3e-05
Batch Size: 32
Weight Decay: 0.01


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warmup Steps: 440


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.397001,0.185407,0.942130,0.943145,0.942130,0.942269
2,0.114718,0.176399,0.949808,0.950146,0.949808,0.949820
3,0.049213,0.218868,0.954927,0.955058,0.954927,0.954958
4,0.020050,0.245680,0.953363,0.953533,0.953363,0.953358
5,0.007743,0.255685,0.955211,0.955269,0.955211,0.955217


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.007743,0.255685,5,0.955211,0.955269,0.955211,0.955217



Final Results:
Accuracy: 0.9552111474477464
Precision: 0.9552688018867542
Recall: 0.9552111474477464
F1 Score: 0.9552165619783101


In [ ]:
results_bert_5 = run_bert_experiment(
    learning_rate=0.00002,
    batch_size=16,
    weight_decay=0.1
)


Starting BERT experiment...
Learning Rate: 2e-05
Batch Size: 16
Weight Decay: 0.1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warmup Steps: 879


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.391638,0.193902,0.945969,0.946590,0.945969,0.946078
2,0.130735,0.198926,0.953931,0.954029,0.953931,0.953921
3,0.058924,0.238057,0.955780,0.955891,0.955780,0.955810
4,0.023870,0.281282,0.950803,0.951506,0.950803,0.950806
5,0.010281,0.281305,0.954785,0.954857,0.954785,0.954774


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.010281,0.238232,5,0.955496,0.955606,0.955496,0.955525



Final Results:
Accuracy: 0.9554955211147448
Precision: 0.9556055385199811
Recall: 0.9554955211147448
F1 Score: 0.9555249695137666


In [ ]:
results_bert_6 = run_bert_experiment(
    learning_rate=0.00003,
    batch_size=16,
    weight_decay=0.1
)


Starting BERT experiment...
Learning Rate: 3e-05
Batch Size: 16
Weight Decay: 0.1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warmup Steps: 879


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.369544,0.202090,0.939855,0.941322,0.939855,0.939947
2,0.130897,0.197651,0.955922,0.955975,0.955922,0.955914
3,0.058137,0.244196,0.954642,0.954736,0.954642,0.954587
4,0.023988,0.265279,0.953505,0.953695,0.953505,0.953491


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.023988,0.197658,4,0.955922,0.955975,0.955922,0.955914



Final Results:
Accuracy: 0.9559220816152424
Precision: 0.9559750544564556
Recall: 0.9559220816152424
F1 Score: 0.9559138143066707


In [ ]:
results_bert_7 = run_bert_experiment(
    learning_rate=0.00002,
    batch_size=32,
    weight_decay=0.1
)


Starting BERT experiment...
Learning Rate: 2e-05
Batch Size: 32
Weight Decay: 0.1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warmup Steps: 440


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.413641,0.182954,0.942557,0.943395,0.942557,0.942673
2,0.119194,0.179797,0.950803,0.951185,0.950803,0.950799
3,0.056506,0.204940,0.954500,0.954663,0.954500,0.954537
4,0.027108,0.225635,0.954927,0.954948,0.954927,0.954929
5,0.013476,0.238618,0.954927,0.954998,0.954927,0.954929


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.013476,0.225654,5,0.954927,0.954948,0.954927,0.954929



Final Results:
Accuracy: 0.9549267737807479
Precision: 0.9549475987390962
Recall: 0.9549267737807479
F1 Score: 0.9549291032668104


In [ ]:
results_bert_8 = run_bert_experiment(
    learning_rate=0.00003,
    batch_size=32,
    weight_decay=0.1
)


Starting BERT experiment...
Learning Rate: 3e-05
Batch Size: 32
Weight Decay: 0.1


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.009189,0.251881,5,0.954358,0.954384,0.954358,0.954346



Final Results:
Accuracy: 0.954358026446751
Precision: 0.9543843001760998
Recall: 0.954358026446751
F1 Score: 0.9543462925361543


In [25]:
bert_result_table = pd.DataFrame({
    "Row": [1, 2, 3, 4, 5, 6, 7, 8],

    "Learning Rate": [
        0.00002, 0.00003, 0.00002, 0.00003,
        0.00002, 0.00003, 0.00002, 0.00003
    ],

    "Batch Size": [16, 16, 32, 32, 16, 16, 32, 32],

    "Weight Decay": [0.01, 0.01, 0.01, 0.01, 0.1, 0.1, 0.1, 0.1]
})

In [26]:
bert_result_table["Accuracy"] = [
    0.9563486421157401,
    0.9572017631167354,
    0.9554955211147448,
    0.9552111474477464,
    0.9554955211147448,
    0.9559220816152424,
    0.9549267737807479,
    0.954358026446751
]

bert_result_table["Precision"] = [
    0.9564647803644236,
    0.9572068364662504,
    0.9554920942339081,
    0.9552688018867542,
    0.9556055385199811,
    0.9559750544564556,
    0.9549475987390962,
    0.9543843001760998
]

bert_result_table["Recall"] = [
    0.9563486421157401,
    0.9572017631167354,
    0.9554955211147448,
    0.9552111474477464,
    0.9554955211147448,
    0.9559220816152424,
    0.9549267737807479,
    0.954358026446751
]

bert_result_table["F1 Score"] = [
    0.9563678825910042,
    0.9571953223373034,
    0.9554833262088749,
    0.9552165619783101,
    0.9555249695137666,
    0.9559138143066707,
    0.9549291032668104,
    0.9543462925361543
]

bert_result_table

,Row,Learning Rate,Batch Size,Weight Decay,Accuracy,Precision,Recall,F1 Score
0,1,0.00002,16,0.01,0.956349,0.956465,0.956349,0.956368
1,2,0.00003,16,0.01,0.957202,0.957207,0.957202,0.957195
2,3,0.00002,32,0.01,0.955496,0.955492,0.955496,0.955483
3,4,0.00003,32,0.01,0.955211,0.955269,0.955211,0.955217
4,5,0.00002,16,0.10,0.955496,0.955606,0.955496,0.955525
5,6,0.00003,16,0.10,0.955922,0.955975,0.955922,0.955914
6,7,0.00002,32,0.10,0.954927,0.954948,0.954927,0.954929
7,8,0.00003,32,0.10,0.954358,0.954384,0.954358,0.954346


In [27]:
bert_result_table[
    ["Accuracy", "Precision", "Recall", "F1 Score"]
] = bert_result_table[
    ["Accuracy", "Precision", "Recall", "F1 Score"]
].round(4)

bert_result_table

,Row,Learning Rate,Batch Size,Weight Decay,Accuracy,Precision,Recall,F1 Score
0,1,0.00002,16,0.01,0.9563,0.9565,0.9563,0.9564
1,2,0.00003,16,0.01,0.9572,0.9572,0.9572,0.9572
2,3,0.00002,32,0.01,0.9555,0.9555,0.9555,0.9555
3,4,0.00003,32,0.01,0.9552,0.9553,0.9552,0.9552
4,5,0.00002,16,0.10,0.9555,0.9556,0.9555,0.9555
5,6,0.00003,16,0.10,0.9559,0.9560,0.9559,0.9559
6,7,0.00002,32,0.10,0.9549,0.9549,0.9549,0.9549
7,8,0.00003,32,0.10,0.9544,0.9544,0.9544,0.9543


In [28]:
best_bert = bert_result_table.loc[
    bert_result_table["F1 Score"].idxmax()
]

print("Best BERT Configuration:")
print(best_bert)

Best BERT Configuration:
Row               2.00000
Learning Rate     0.00003
Batch Size       16.00000
Weight Decay      0.01000
Accuracy          0.95720
Precision         0.95720
Recall            0.95720
F1 Score          0.95720
Name: 1, dtype: float64


In [29]:
best_accuracy = bert_result_table.loc[
    bert_result_table["Accuracy"].idxmax()
]

print("Best Accuracy Configuration:")
print(best_accuracy)

Best Accuracy Configuration:
Row               2.00000
Learning Rate     0.00003
Batch Size       16.00000
Weight Decay      0.01000
Accuracy          0.95720
Precision         0.95720
Recall            0.95720
F1 Score          0.95720
Name: 1, dtype: float64


In [33]:
bert_result_table.to_csv(
    "dataset1_bert_results.csv",
    index=False
)

print("Dataset 1 BERT results saved successfully.")

Dataset 1 BERT results saved successfully.


In [32]:
import os

print(os.path.exists("dataset1_bert_results.csv"))

True
